# 02 – Exploration
Exploratory analysis of DE-LU day-ahead prices and covariate forecasts on the pre-test period (2023-01-01 → 2026-03-31).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config as cfg

df = pd.read_parquet(cfg.DATASET_PATH)
pre = df.loc[:"2026-03-31"]   # pre-test period only

print(pre.shape)
print(pre.index.min(), "→", pre.index.max())
pre.head()


## 1. Series overview and descriptive statistics

In [ ]:
cols = ["price", "load_fc", "wind_fc", "solar_fc"]
labels = {
    "price":    "Day-ahead price\n[EUR/MWh]",
    "load_fc":  "Load forecast\n[GW]",
    "wind_fc":  "Wind forecast\n(on+offshore) [GW]",
    "solar_fc": "Solar forecast\n[GW]",
}

fig, axes = plt.subplots(4, 1, sharex=True, figsize=(14, 10))

for ax, col in zip(axes, cols):
    ax.plot(pre.index, pre[col], lw=0.3, alpha=0.5, label="hourly")
    daily = pre[col].resample("D").mean()
    ax.plot(daily.index, daily, lw=1, color="C1", label="daily mean")
    ax.set_ylabel(labels[col])
    ax.legend(loc="upper right", fontsize=8)

axes[0].axhline(0, color="k", lw=0.5)   # zero line: makes negative prices visible
fig.suptitle("DE-LU day-ahead price and day-ahead forecasts (published on D−1), pre-test period")
fig.tight_layout()
fig.savefig(cfg.FIGURES_DIR / "series_overview.png", dpi=150)
plt.show()

In [ ]:
pre[cols].describe().round(1)

In [ ]:
p = pre["price"]

print(f"Min price: {p.min():.1f} EUR/MWh on {p.idxmin()}")
print(f"Max price: {p.max():.1f} EUR/MWh on {p.idxmax()}")
print(f"Negative-price hours: {(p < 0).sum()} ({(p < 0).mean():.1%} of all hours)")
print("\nPrice standard deviation per year [EUR/MWh]:")
print(p.groupby(p.index.year).std().round(1))


**Observations.** Prices mostly lie between 67 and 114 EUR/MWh (interquartile range), with rare but extreme values in both directions: −500 EUR/MWh (market floor) on Sunday 2 July 2023 at 14:00 (high solar, low demand) and 936 EUR/MWh on 12 December 2024 at 17:00 (winter evening, almost no wind). About 5 % of hours have negative prices, and volatility comes in clusters. Load shows yearly and weekly seasonality, wind is irregular and weather-driven, and solar is strongly seasonal and growing year after year.

## 2. Daily and weekly profiles


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [3, 1]})

# Price distribution for each hour of the day
by_hour = [pre.loc[pre["hour"] == h, "price"] for h in range(24)]
axes[0].boxplot(by_hour, showfliers=False)
axes[0].set_xticks(range(1, 25), range(24))
axes[0].set_xlabel("Hour of day (Berlin time)")
axes[0].set_ylabel("Price [EUR/MWh]")
axes[0].set_title("Price by hour of day")

# Price distribution for each day of the week
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
by_dow = [pre.loc[pre["dow"] == d, "price"] for d in range(7)]
axes[1].boxplot(by_dow, showfliers=False)
axes[1].set_xticks(range(1, 8), days)
axes[1].set_title("Price by day of week")

for ax in axes:
    ax.axhline(0, color="k", lw=0.5)

fig.suptitle("Price profiles, pre-test period (outliers hidden)")
fig.tight_layout()
fig.savefig(cfg.FIGURES_DIR / "price_profiles_boxplots.png", dpi=150)
plt.show()


In [ ]:
profile = pre.groupby(["non_working_day", "hour"])["price"].mean().unstack(0)
profile.columns = ["Working day", "Non-working day (weekend / holiday)"]

ax = profile.plot(figsize=(10, 4), marker="o", ms=3)
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("Hour of day (Berlin time)")
ax.set_ylabel("Mean price [EUR/MWh]")
ax.set_title("Mean daily price profile: working vs non-working days")
ax.set_xticks(range(24))
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / "profile_working_vs_nonworking.png", dpi=150)
plt.show()

profile.round(0).T


**Observations.** Prices show a clear daily cycle: a morning peak (7–8 h), an evening peak (19–20 h) and a midday dip (13–14 h) caused by solar generation, where dispersion and negative prices are highest. Weekends are cheaper, Sunday most of all. On non-working days the morning peak disappears and midday prices drop much further, which supports using `non_working_day` as a covariate and a daily season s = 24.